# Reading a CRISPR-Cas13 lateral flow strip

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cc0231/Lateral-Flow-Interpretation-Code/blob/main/notebooks/demo.ipynb)

Segment the strip, then classify the mask as positive or negative.

```
cropped strip -> segmentation (MnUV3 @ 256) -> mask -> classifier (@ 256) -> POSITIVE / NEGATIVE
```

**Input is an already-cropped 512x512 strip region, not a full phone photo.**
Cropping was done by hand and there is no detector in this repo.

Paper: [Xue et al., *Sensors & Diagnostics*, 2024](https://doi.org/10.1039/d4sd00314d)

## Setup

Skip the clone if you are running this locally inside the repo.

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !git clone -q https://github.com/cc0231/Lateral-Flow-Interpretation-Code.git
    os.chdir('Lateral-Flow-Interpretation-Code')
    !pip install -q albumentations

print('working dir:', os.getcwd())

## Weights

Not in the repo (too large) - they come from the GitHub Release. Each file is
weights-only and loads under `weights_only=True`.

In [ ]:
from pathlib import Path

Path('weights').mkdir(exist_ok=True)
BASE = 'https://github.com/cc0231/Lateral-Flow-Interpretation-Code/releases/latest/download'
for f in ('mnuv3_seg_train_no_test_no.pth', 'classifier.pth'):
    if not Path('weights', f).exists():
        !wget -q -P weights {BASE}/{f}

!ls -lh weights/

## Run it

On the bundled sample strips.

In [ ]:
import torch
from src.models import MnUV3, Network
from src.predict import load_weights, predict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
seg = load_weights(MnUV3(), 'weights/mnuv3_seg_train_no_test_no.pth', device)
cls = load_weights(Network(), 'weights/classifier.pth', device)
print('loaded on', device)

In [ ]:
images = sorted(Path('sample_data/images').glob('*.jpg'))[:6]

results = [predict(p, seg, cls, device) for p in images]
for r in results:
    print(f"{Path(r['image']).name:<16} P(pos)={r['probability_positive']:.3f}  {r['call']}")

## Look at what it segmented

The mask is what the classifier actually sees: one band is negative, two is positive.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, len(results), figsize=(2.2 * len(results), 5))
for ax, r in zip(axes[0], results):
    ax.imshow(Image.open(r['image'])); ax.set_title(Path(r['image']).name, fontsize=8); ax.axis('off')
for ax, r in zip(axes[1], results):
    ax.imshow(r['mask'], cmap='gray')
    ax.set_title(f"{r['call']}  {r['probability_positive']:.2f}", fontsize=8); ax.axis('off')
axes[0, 0].set_ylabel('strip'); axes[1, 0].set_ylabel('mask')
plt.tight_layout(); plt.show()

## Your own strip

Crop the strip region out of the photo first, then:

```python
r = predict('my_strip.jpg', seg, cls, device)
print(r['call'], r['probability_positive'])
```

Or from a shell:

```bash
python -m src.predict --image my_strip.jpg \n    --seg-weights weights/mnuv3_seg_train_no_test_no.pth \n    --cls-weights weights/classifier.pth --arch mnuv3
```